In [ ]:
from google.colab import drive
drive.mount("/content/gdrive/", force_remount=True)
import pandas as pd
import numpy as np
import os
import openpyxl
from tabulate import tabulate
from openpyxl.styles import PatternFill
import time
from datetime import timedelta

Mounted at /content/gdrive/


In [ ]:
def recommend_list(user, N, k, training_df, testing_df, User_Average_DataFrame, sim_path, real_pre_DataFrame, sim_item_path):
    user_col = []
    user_col_rmse = []
    item_col = []
    rating_real_col = []
    rating_predict_col = []
    neighbors_positions_col = []
    total_rating = 0
    num_neighbors = 0

    # Lấy ra độ tương đồng của người dùng
    user_df = pd.read_excel(r"{0}/User{1}.xlsx".format(sim_path, user))

    # Danh sách các item cần dự đoán
    predict_list = testing_df.Item[testing_df.User == user].to_list()
    R_u_tb = User_Average_DataFrame.UserAvg[User_Average_DataFrame.User == user].values[0]

    # Vòng lặp item
    for j in predict_list:
        # Danh sách những người dùng có đánh giá item j
        user_j = training_df.User[training_df.Item == j].to_list()
        user_j_top_k = user_df.User[user_df.User.isin(user_j) == True][:k].to_list()  # Lấy top láng giềng có rating item j

        # Lưu lại vị trí của các user_j_top_k trong user_df
        neighbor_positions = user_df[user_df.User.isin(user_j_top_k)].index.tolist()
        neighbors_positions_col.append(neighbor_positions)

        for neighbor in user_j_top_k:  # Vòng lặp tính tổng rating của các láng giềng cho item j
            neighbor_rating = training_df.Rating[(training_df.User == neighbor) & (training_df.Item == j)].values[0]
            total_rating += neighbor_rating
            num_neighbors += 1
        if num_neighbors > 0:
            ave_pred_rating = total_rating / num_neighbors
        else:
            ave_pred_rating = R_u_tb  # Sử dụng rating trung bình của người dùng nếu không có láng giềng nào
        num_neighbors = 0
        total_rating = 0

        # Tìm rating trong test
        rate_real = testing_df.Rating[testing_df.User == user][testing_df.Item == j].values[0]
        # Thêm id item j vào cột item
        user_col.append(user)
        item_col.append(j)

        # Thêm sim vào cột đánh giá
        rating_real_col.append(rate_real)
        rating_predict_col.append(ave_pred_rating)
        user_col_rmse.append(np.sqrt(pow(ave_pred_rating - rate_real, 2)))

    df = pd.DataFrame(data={'User': user_col, 'Item': item_col, 'Rating_real': rating_real_col,
                            'Rating_predict': rating_predict_col, 'RMSE': user_col_rmse,
                            'Neighbors_Positions': neighbors_positions_col})
    df = df.sort_values(by=['Rating_predict'], axis=0, ascending=False, ignore_index=True)

    real_pre_DataFrame = pd.concat([real_pre_DataFrame, df])
    # Sắp xếp đánh giá dự đoán theo giá trị đánh giá

    # Lấy danh sách item từ 1 tới N
    df = df.iloc[:N]

    # Trả về danh sách N item
    return df.Item.to_list(), real_pre_DataFrame

In [ ]:
def revaluation(N, k, sim_path, testing_df, training_df, first, last, sim_item_path):
    # Khai báo trước những thước đo đánh giá thành các cột
    real_pre_df = pd.DataFrame()
    User_col = []
    I_N_col = []
    I_te_col = []
    intersection_col = []
    I_N_len_col = []
    I_te_len_col = []
    intersection_len_col = []
    Prec_u_N = []
    Rec_u_N = []
    F1_u_N = []

    testing_user = testing_df.User[testing_df.User >= 0].to_list()
    testing_user = list(dict.fromkeys(testing_user))
    # Vòng lặp người dùng
    for i in range(first, last + 1):
        if i in testing_user and i in training_df.User.to_list():
            # Danh sách item hệ thống recommend cho người dùng
            I_N, real_pre_df = recommend_list(i, N, k, training_df, testing_df, user_avg_df, sim_path, real_pre_df, sim_item_path)
            # Danh sách item trong file test của người dùng
            I_te = testing_df.Item[testing_df.User == i][testing_df.Rating > 3].to_list()
            # Phần giao giữa danh sách item
            intersection = [value for value in I_N if value in I_te]
            try:
                # Tính thước đo đánh giá
                Prec = abs(len(intersection)) / abs(len(I_N))
                Rec = abs(len(intersection)) / abs(len(I_te))
                F1 = 2 * (Prec * Rec) / (Prec + Rec)
            except:
                # Trường hợp các list này rỗng
                if intersection == [] or I_te == [] or I_N == []:
                    Prec = 0
                    Rec = 0
                    F1 = 0
        else:
            continue
        # Thêm id người dùng, các độ đo vào các cột
        User_col.append(i)
        I_N_col.append(' '.join(str(e) for e in I_N))
        I_te_col.append(' '.join(str(e) for e in I_te))
        intersection_col.append(' '.join(str(e) for e in intersection))
        I_N_len_col.append(len(I_N))
        I_te_len_col.append(len(I_te))
        intersection_len_col.append(len(intersection))
        Prec_u_N.append(Prec)
        Rec_u_N.append(Rec)
        F1_u_N.append(F1)

    # Tạo dataframe
    df = pd.DataFrame(data={'User': User_col, 'Tập tư vấn (C)': I_N_col, 'Tập đúng (T)': I_te_col, 'Giao TC': intersection_col,
                            'C_length': I_N_len_col, 'T_length': I_te_len_col, 'TC_length': intersection_len_col,
                            'Precision': Prec_u_N, 'Recall': Rec_u_N, 'F1': F1_u_N})

    # Tính tổng F1
    if df.C_length.sum() > 0:
        pre = df.TC_length.sum() / df.C_length.sum()
    else:
        pre = 0

    if df.T_length.sum() > 0:
        recall = df.TC_length.sum() / df.T_length.sum()
    else:
        recall = 0
    if pre + recall > 0:
        f1 = 2 * pre * recall / (pre + recall)
    else:
        f1 = 0

    df_all = pd.DataFrame(data={'Precision_system': [pre], 'Recall_system': [recall], 'F1': [f1]})
    # Tính tổng RMSE
    all = np.sqrt(real_pre_df.RMSE.sum() / real_pre_df.RMSE.notnull().sum())

    # Thêm cột position_neighbor
    real_pre_df['Position'] = real_pre_df['Neighbors_Positions']
    real_pre_df_all = pd.DataFrame(data={'Position': real_pre_df['Position'].tolist(), 'RMSE_system': [all]*len(real_pre_df)})

    # Xuất file kết quả
    result_path = os.path.join(sim_path[0:-8], 'Result_Ave_Prediction_Position')
    if not os.path.exists(result_path):
        os.makedirs(result_path)

    # Chỉ cần để ý F1 score và RMSE
    file_path = os.path.join(result_path, "{}_N{}_k{}_Ave_position.xlsx".format(name, N, k))
    with pd.ExcelWriter(file_path) as writer:
        df.to_excel(writer, sheet_name='F1_RMSE', index=False)  # F1
        df_all.to_excel(writer, sheet_name='F1_RMSE', startcol=10, index=False)  # F1 tổng
        real_pre_df.to_excel(writer, sheet_name='F1_RMSE', startcol=13, index=False)  # RMSE và Position
        real_pre_df_all.to_excel(writer, sheet_name='F1_RMSE', startcol=18, index=False)  # RMSE tổng

In [ ]:
def convert_seconds_to_time(seconds):
    # Chuyển số giây thành đối tượng timedelta
    duration = timedelta(seconds=seconds)

    # Tính toán các thành phần ngày, giờ, phút và giây
    days = duration.days
    hours, remainder = divmod(duration.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    # Tạo chuỗi định dạng ngày:giờ:phút:giây
    formatted_time = "{:02}d:{:02}h:{:02}m:{:02}s".format(days, hours, minutes, seconds)

    return formatted_time

In [ ]:
#name = 'dlmau'
name = 'SweetRS'
# name = 'MusicInCar'

#đọc file training, testing
training_df = pd.read_excel(f'/content/gdrive/MyDrive/ColabNotebooks/Dataset/{name}/{name}_training.xlsx', dtype={'User': int, 'Item': int})
testing_df = pd.read_excel(f'/content/gdrive/MyDrive/ColabNotebooks/Dataset/{name}/{name}_testing.xlsx', dtype={'User': int, 'Item': int})
#Đọc file User_Average
user_avg_df = pd.read_excel(f'/content/gdrive/MyDrive/ColabNotebooks/Dataset/{name}/{name}_User_Average.xlsx')
user_avg_df = user_avg_df.rename(columns={'User_Average': 'UserAvg'}, errors="raise")

#số lượng user
n_user = np.max(training_df.User)
N = 5  # 5 7 10
k = 20 # 3 5 10 20

#đo thời gian
start_time = time.time()
revaluation(N, k, r"/content/gdrive/MyDrive/ColabNotebooks/Dataset/{}/SimUser/".format(name), testing_df, training_df, 1, n_user, r"/content/gdrive/MyDrive/ColabNotebooks/Dataset/{}/SimItem/".format(name))
end_time = time.time()

#đọc file kết quả
data = pd.read_excel(r'/content/gdrive/MyDrive/ColabNotebooks/Dataset/{0}/Result_Ave_Prediction_Position/{0}_N{1}_k{2}_Ave_position.xlsx'.format(name, N, k))

#tạo dữ liệu cho time
time_all = pd.DataFrame({'Time_all': [convert_seconds_to_time(end_time - start_time)]})

#Thêm cột time vào file excel
file_path = os.path.join(r'/content/gdrive/MyDrive/ColabNotebooks/Dataset/{}/Result_Ave_Prediction_Position'.format(name), "{0}_N{1}_k{2}_Ave_position.xlsx".format(name, N, k))
with pd.ExcelWriter(file_path) as writer:
  data.to_excel(writer, sheet_name = 'F1_RMSE', index = False)
  time_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 20, index = False)